In [ ]:
# ============================================================
# Carbon-chain full-accuracy SVD ratios
# One GIF, 8 threads, memory-conscious version
# ============================================================

# IMPORTANT:
# In Jupyter, restart the kernel before running this cell.
# These environment variables should be set before importing NumPy/SciPy.

import os

os.environ["OMP_NUM_THREADS"] = "8"
os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"
os.environ["VECLIB_MAXIMUM_THREADS"] = "8"
os.environ["NUMEXPR_NUM_THREADS"] = "8"


from pathlib import Path
import re
import csv
import gc
import warnings

import h5py
import numpy as np
import scipy.sparse as sp
from scipy.linalg import svd

import matplotlib.pyplot as plt
from PIL import Image


# ============================================================
# Configuration
# ============================================================

H5_PATH = Path("/scratch/yimili/matrices/hdf5/carbon-chain.h5")

OUT_DIR = Path("/scratch/yimili/non-normal/carbon-chain")
FRAME_DIR = OUT_DIR / "frames"

GIF_PATH = OUT_DIR / "non_normal_shift.gif"

RATIO_MATRIX_PATH = OUT_DIR / "ratio_matrix.npy"
LOG_CUMULATIVE_MATRIX_PATH = OUT_DIR / "log_cumulative_ratio_matrix.npy"
SINGULAR_VALUE_MATRIX_PATH = OUT_DIR / "singular_value_matrix.npy"
EIGENVALUE_MAGNITUDE_MATRIX_PATH = OUT_DIR / "eigenvalue_magnitude_matrix.npy"
INDEX_PATH = OUT_DIR / "ratio_matrix_indices.npy"
NNZ_PATH = OUT_DIR / "nnz_by_index.npy"

CSV_PATH = OUT_DIR / "singular_eigenvalue_ratios.csv"

# Specify indices here.
# Examples:
# INDEX_RANGE = range(0, 401)
# INDEX_RANGE = range(40, 81)
# INDEX_RANGE = [60]
INDEX_RANGE = range(0, 401)

# Thresholds for counting ratios outside 1 +/- DIFF.
DIFF = 0.01

# Use "linear" or "log".
RATIO_Y_SCALE = "linear"

# GIF settings.
GIF_FPS = 4
PING_PONG = False

# Plot settings.
DPI = 120
CLEAN_OLD_FRAMES = True

# Full accuracy.
# For real matrices, this uses float64.
# For complex matrices, the dense matrix is complex128.
DTYPE_REAL = np.float64

# CSV can be large and slow. The .npy files are better for large runs.
SAVE_CSV = True


# ============================================================
# Prepare output folders
# ============================================================

OUT_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DIR.mkdir(parents=True, exist_ok=True)

if CLEAN_OLD_FRAMES:
    for old_frame in FRAME_DIR.glob("E_*.png"):
        old_frame.unlink()


# ============================================================
# Helper functions
# ============================================================

def discover_indices(h5_file):
    """
    Return all numerical indices for top-level groups named E_<integer>
    that contain both M and spectrum.
    """
    pattern = re.compile(r"^E_(\d+)$")
    found = []

    for name in h5_file.keys():
        match = pattern.fullmatch(name)

        if match is None:
            continue

        if not isinstance(h5_file[name], h5py.Group):
            continue

        if "M" in h5_file[name] and "spectrum" in h5_file[name]:
            found.append(int(match.group(1)))

    return sorted(found)


def load_csc_matrix(h5_file, group_name):
    """
    Load CSC matrix stored under E_<index>/M.
    """
    grp = h5_file[f"{group_name}/M"]

    data = grp["data"][...]
    row_indices = grp["indices"][...]
    column_pointer = grp["indptr"][...]

    shape = None

    if "shape" in grp.attrs:
        shape = tuple(
            int(value)
            for value in np.asarray(grp.attrs["shape"]).ravel()
        )

    elif "shape" in grp:
        shape = tuple(
            int(value)
            for value in np.asarray(grp["shape"][...]).ravel()
        )

    if shape is None:
        n_cols = len(column_pointer) - 1
        shape = (n_cols, n_cols)

        print(
            f"[warning] {group_name}: no shape stored; "
            f"assuming square matrix {shape}"
        )

    if len(shape) != 2:
        raise ValueError(f"{group_name}: invalid matrix shape: {shape}")

    return sp.csc_matrix(
        (data, row_indices, column_pointer),
        shape=shape,
    )


def sparse_to_dense_for_svd(M):
    """
    Convert sparse matrix to dense Fortran-contiguous array.

    Full SVD needs a dense matrix anyway when all singular values are required.
    Fortran order is preferred by LAPACK and can avoid extra internal copies.
    """
    A = M.toarray()

    if np.isrealobj(A):
        A = np.asarray(A, dtype=np.float64, order="F")
    else:
        A = np.asarray(A, dtype=np.complex128, order="F")

    return A


# ============================================================
# Select indices and determine matrix size
# ============================================================

with h5py.File(H5_PATH, "r") as h5_file:

    available_indices = discover_indices(h5_file)

    if not available_indices:
        raise RuntimeError(
            "No groups matching E_<integer> with both M and spectrum were found."
        )

    if INDEX_RANGE is None:
        selected_indices = available_indices
    else:
        requested_indices = set(INDEX_RANGE)
        selected_indices = [
            index
            for index in available_indices
            if index in requested_indices
        ]

        missing_indices = sorted(requested_indices.difference(selected_indices))

        if missing_indices:
            shown_missing = missing_indices[:20]
            suffix = " ..." if len(missing_indices) > 20 else ""
            print(
                "[warning] Some requested indices are not present: "
                f"{shown_missing}{suffix}"
            )

    selected_indices = sorted(selected_indices)

    if not selected_indices:
        raise RuntimeError("No indices were selected.")

    first_group = f"E_{selected_indices[0]}"
    first_spectrum = np.asarray(
        h5_file[f"{first_group}/spectrum"][...]
    ).squeeze()

    n = first_spectrum.size

    # This memory-efficient version assumes same size for all selected matrices.
    for index in selected_indices:
        group_name = f"E_{index}"
        spectrum_size = np.asarray(
            h5_file[f"{group_name}/spectrum"][...]
        ).squeeze().size

        if spectrum_size != n:
            raise ValueError(
                f"{group_name} has spectrum size {spectrum_size}, "
                f"but first selected matrix has size {n}. "
                "This code assumes all selected matrices have the same size."
            )

num_indices = len(selected_indices)

print(f"Selected {num_indices} matrices.")
print(f"Matrix/spectrum size: {n}")
print(f"Using 8 BLAS/LAPACK threads per SVD.")


# ============================================================
# Create disk-backed arrays
# ============================================================

ratio_matrix = np.lib.format.open_memmap(
    RATIO_MATRIX_PATH,
    mode="w+",
    dtype=np.float64,
    shape=(num_indices, n),
)

log_cumulative_ratio_matrix = np.lib.format.open_memmap(
    LOG_CUMULATIVE_MATRIX_PATH,
    mode="w+",
    dtype=np.float64,
    shape=(num_indices, n),
)

singular_value_matrix = np.lib.format.open_memmap(
    SINGULAR_VALUE_MATRIX_PATH,
    mode="w+",
    dtype=np.float64,
    shape=(num_indices, n),
)

eigenvalue_magnitude_matrix = np.lib.format.open_memmap(
    EIGENVALUE_MAGNITUDE_MATRIX_PATH,
    mode="w+",
    dtype=np.float64,
    shape=(num_indices, n),
)

processed_indices = np.asarray(selected_indices, dtype=np.int64)
np.save(INDEX_PATH, processed_indices)

nnz_array = np.full(num_indices, -1, dtype=np.int64)


# ============================================================
# First pass:
# compute all singular values and ratios, write directly to disk
# ============================================================

global_ratio_min = np.inf
global_ratio_max = -np.inf

global_log_cum_min = np.inf
global_log_cum_max = -np.inf

valid_rows = np.zeros(num_indices, dtype=bool)

with h5py.File(H5_PATH, "r") as h5_file:

    for row, index in enumerate(selected_indices):

        group_name = f"E_{index}"

        M = None
        A = None
        spectrum = None
        eigenvalue_magnitudes = None
        singular_values = None
        ratio = None
        log_cumulative_ratio = None

        try:
            # ------------------------------------------------
            # Load saved eigenvalue spectrum
            # ------------------------------------------------

            spectrum = np.asarray(
                h5_file[f"{group_name}/spectrum"][...]
            ).squeeze()

            if spectrum.ndim != 1:
                raise ValueError(
                    f"spectrum has shape {spectrum.shape}; expected 1D array"
                )

            eigenvalue_magnitudes = np.sort(
                np.asarray(np.abs(spectrum), dtype=np.float64)
            )[::-1]

            if not np.all(np.isfinite(eigenvalue_magnitudes)):
                raise ValueError("saved spectrum contains NaN or Inf")

            if np.any(eigenvalue_magnitudes <= 0):
                raise ValueError("one or more eigenvalue magnitudes are zero")

            # ------------------------------------------------
            # Load sparse matrix and convert to dense
            # ------------------------------------------------

            M = load_csc_matrix(h5_file, group_name)

            if M.shape[0] != M.shape[1]:
                raise ValueError(f"matrix is not square: shape={M.shape}")

            if M.shape[0] != n:
                raise ValueError(
                    f"matrix shape {M.shape} does not match spectrum size {n}"
                )

            nnz_array[row] = M.nnz

            A = sparse_to_dense_for_svd(M)

            if not np.all(np.isfinite(A)):
                raise ValueError("dense matrix contains NaN or Inf")

            # ------------------------------------------------
            # Full-accuracy dense SVD, singular values only
            # ------------------------------------------------

            singular_values = svd(
                A,
                compute_uv=False,
                full_matrices=False,
                overwrite_a=True,
                check_finite=False,
                lapack_driver="gesdd",
            )

            singular_values = np.asarray(singular_values, dtype=np.float64)

            if singular_values.size != n:
                raise ValueError(
                    f"found {singular_values.size} singular values, expected {n}"
                )

            if not np.all(np.isfinite(singular_values)):
                raise ValueError("singular values contain NaN or Inf")

            if np.any(singular_values <= 0):
                raise ValueError("one or more singular values are zero")

            # ------------------------------------------------
            # Compute ratios
            # ------------------------------------------------

            ratio = singular_values / eigenvalue_magnitudes

            log_cumulative_ratio = np.cumsum(
                np.log(singular_values) - np.log(eigenvalue_magnitudes)
            )

            if not np.all(np.isfinite(ratio)):
                raise ValueError("ratio contains NaN or Inf")

            if not np.all(np.isfinite(log_cumulative_ratio)):
                raise ValueError("log cumulative ratio contains NaN or Inf")

            # ------------------------------------------------
            # Save row directly to disk-backed arrays
            # ------------------------------------------------

            ratio_matrix[row, :] = ratio
            log_cumulative_ratio_matrix[row, :] = log_cumulative_ratio
            singular_value_matrix[row, :] = singular_values
            eigenvalue_magnitude_matrix[row, :] = eigenvalue_magnitudes

            valid_rows[row] = True

            # Flush periodically so progress is saved.
            ratio_matrix.flush()
            log_cumulative_ratio_matrix.flush()
            singular_value_matrix.flush()
            eigenvalue_magnitude_matrix.flush()

            # Track global plot limits.
            global_ratio_min = min(global_ratio_min, float(np.min(ratio)))
            global_ratio_max = max(global_ratio_max, float(np.max(ratio)))

            global_log_cum_min = min(
                global_log_cum_min,
                float(np.min(log_cumulative_ratio)),
            )

            global_log_cum_max = max(
                global_log_cum_max,
                float(np.max(log_cumulative_ratio)),
            )

            print(
                f"[{row + 1:>4}/{num_indices}] "
                f"{group_name}: "
                f"shape={M.shape}, "
                f"nnz={M.nnz}, "
                f"sigma_max={singular_values[0]:.6e}, "
                f"sigma_min={singular_values[-1]:.6e}, "
                f"endpoint={log_cumulative_ratio[-1]:+.3e}"
            )

        except Exception as error:
            warnings.warn(f"Skipping {group_name}: {error}")

            ratio_matrix[row, :] = np.nan
            log_cumulative_ratio_matrix[row, :] = np.nan
            singular_value_matrix[row, :] = np.nan
            eigenvalue_magnitude_matrix[row, :] = np.nan

            ratio_matrix.flush()
            log_cumulative_ratio_matrix.flush()
            singular_value_matrix.flush()
            eigenvalue_magnitude_matrix.flush()

        finally:
            # ------------------------------------------------
            # Explicit cleanup after each matrix
            # ------------------------------------------------

            del log_cumulative_ratio
            del ratio
            del singular_values
            del eigenvalue_magnitudes
            del spectrum
            del A
            del M

            gc.collect()


np.save(NNZ_PATH, nnz_array)

if not np.any(valid_rows):
    raise RuntimeError("No matrices were successfully processed.")


# ============================================================
# Compute global plot limits
# ============================================================

x_right = max(n, 2)

ratio_low = min(
    global_ratio_min,
    1.0 - DIFF,
    1.0,
)

ratio_high = max(
    global_ratio_max,
    1.0 + DIFF,
    1.0,
)

if RATIO_Y_SCALE == "log":

    if ratio_low <= 0:
        raise ValueError("A logarithmic ratio axis requires positive ratios.")

    log_low = np.log10(ratio_low)
    log_high = np.log10(ratio_high)

    log_span = log_high - log_low
    log_padding = max(0.05 * log_span, 0.02)

    ratio_ylim = (
        10.0 ** max(log_low - log_padding, -300.0),
        10.0 ** min(log_high + log_padding, 300.0),
    )

elif RATIO_Y_SCALE == "linear":

    ratio_span = ratio_high - ratio_low
    ratio_padding = max(0.05 * ratio_span, 0.02)

    ratio_ylim = (
        max(0.0, ratio_low - ratio_padding),
        ratio_high + ratio_padding,
    )

else:
    raise ValueError("RATIO_Y_SCALE must be either 'linear' or 'log'.")


cumulative_low = min(global_log_cum_min, 0.0)
cumulative_high = max(global_log_cum_max, 0.0)

cumulative_span = cumulative_high - cumulative_low
cumulative_padding = max(0.05 * cumulative_span, 0.05)

cumulative_ylim = (
    cumulative_low - cumulative_padding,
    cumulative_high + cumulative_padding,
)


# ============================================================
# Second pass:
# read saved arrays one row at a time and create frames
# ============================================================

frame_paths = []
k = np.arange(1, n + 1)

for frame_number, index in enumerate(selected_indices, start=1):

    row = frame_number - 1

    if not valid_rows[row]:
        print(f"[warning] Skipping frame for E_{index}; row is invalid.")
        continue

    ratio = np.asarray(ratio_matrix[row, :])
    log_cumulative_ratio = np.asarray(
        log_cumulative_ratio_matrix[row, :]
    )

    if np.any(np.isnan(ratio)) or np.any(np.isnan(log_cumulative_ratio)):
        print(f"[warning] Skipping frame for E_{index}; data contains NaN.")
        continue

    fig, (ax1, ax2) = plt.subplots(
        1,
        2,
        figsize=(13, 5),
        constrained_layout=True,
    )

    # --------------------------------------------------------
    # Plot 1: pointwise ratio
    # --------------------------------------------------------

    ax1.plot(
        k,
        ratio,
        marker="o",
        markersize=1.5,
        linewidth=0.8,
    )

    ax1.axhline(
        1.0,
        color="gray",
        linestyle="--",
        linewidth=1.0,
        label="ratio = 1",
    )

    ax1.axhline(
        1.0 + DIFF,
        color="gray",
        linestyle=":",
        linewidth=0.9,
    )

    ax1.axhline(
        1.0 - DIFF,
        color="gray",
        linestyle=":",
        linewidth=0.9,
    )

    ax1.set_yscale(RATIO_Y_SCALE)
    ax1.set_xlim(1, x_right)
    ax1.set_ylim(*ratio_ylim)

    ax1.set_xlabel("rank i, descending order")
    ax1.set_ylabel(r"$\sigma_i / |\lambda_i|$")
    ax1.set_title("Pointwise singular-value/eigenvalue ratio")
    ax1.grid(alpha=0.3, which="both")
    ax1.legend(loc="best")

    # --------------------------------------------------------
    # Plot 2: cumulative product ratio in log form
    # --------------------------------------------------------

    ax2.plot(
        k,
        log_cumulative_ratio,
        linewidth=1.0,
    )

    ax2.axhline(
        0.0,
        color="gray",
        linestyle="--",
        linewidth=1.0,
    )

    ax2.set_xlim(1, x_right)
    ax2.set_ylim(*cumulative_ylim)

    ax2.set_xlabel("k")

    ax2.set_ylabel(
        r"$\log\left("
        r"\prod_{i\leq k}\sigma_i"
        r"\,/\,"
        r"\prod_{i\leq k}|\lambda_i|"
        r"\right)$"
    )

    ax2.set_title("Cumulative product ratio, logarithmic form")
    ax2.grid(alpha=0.3)

    # --------------------------------------------------------
    # Frame title
    # --------------------------------------------------------

    number_above = int(np.count_nonzero(ratio > 1.0 + DIFF))
    number_below = int(np.count_nonzero(ratio < 1.0 - DIFF))

    fig.suptitle(
        f"E_{index}   "
        f"frame {len(frame_paths) + 1}   "
        f"n={n}   "
        f"nnz={nnz_array[row]}   "
        f"> {1.0 + DIFF:.2f}: {number_above}   "
        f"< {1.0 - DIFF:.2f}: {number_below}   "
        f"endpoint={log_cumulative_ratio[-1]:+.2e}",
        fontsize=12,
    )

    frame_path = FRAME_DIR / f"E_{index:06d}.png"

    fig.savefig(
        frame_path,
        dpi=DPI,
        facecolor="white",
    )

    plt.close(fig)

    frame_paths.append(frame_path)

    del ratio
    del log_cumulative_ratio
    gc.collect()


if not frame_paths:
    raise RuntimeError("No frames were created; cannot make GIF.")


# ============================================================
# Create one final GIF using Pillow
# ============================================================

gif_sequence = list(frame_paths)

if PING_PONG and len(frame_paths) > 2:
    gif_sequence += frame_paths[-2:0:-1]

duration_ms = int(1000 / GIF_FPS)

first_frame = Image.open(gif_sequence[0]).convert(
    "P",
    palette=Image.ADAPTIVE,
)

append_frames = []

try:
    for frame_path in gif_sequence[1:]:
        frame = Image.open(frame_path).convert(
            "P",
            palette=Image.ADAPTIVE,
        )
        append_frames.append(frame)

    first_frame.save(
        GIF_PATH,
        save_all=True,
        append_images=append_frames,
        duration=duration_ms,
        loop=0,
    )

finally:
    first_frame.close()

    for frame in append_frames:
        frame.close()


# ============================================================
# Optional CSV export
# ============================================================

if SAVE_CSV:

    print()
    print("Writing CSV. This may take a little while...")

    with open(CSV_PATH, "w", newline="") as csv_file:

        writer = csv.writer(csv_file)

        writer.writerow(
            [
                "matrix_index",
                "group_name",
                "rank",
                "singular_value",
                "eigenvalue_magnitude",
                "ratio_sigma_over_abs_lambda",
                "log_cumulative_ratio",
            ]
        )

        for row, index in enumerate(selected_indices):

            if not valid_rows[row]:
                continue

            for rank in range(n):

                writer.writerow(
                    [
                        index,
                        f"E_{index}",
                        rank + 1,
                        singular_value_matrix[row, rank],
                        eigenvalue_magnitude_matrix[row, rank],
                        ratio_matrix[row, rank],
                        log_cumulative_ratio_matrix[row, rank],
                    ]
                )


# ============================================================
# Final summary
# ============================================================

print()
print("Done.")
print()
print("Saved one GIF to:")
print(GIF_PATH)
print()
print("Saved PNG frames to:")
print(FRAME_DIR)
print()
print("Saved arrays to:")
print(RATIO_MATRIX_PATH)
print(LOG_CUMULATIVE_MATRIX_PATH)
print(SINGULAR_VALUE_MATRIX_PATH)
print(EIGENVALUE_MAGNITUDE_MATRIX_PATH)
print(INDEX_PATH)
print(NNZ_PATH)

if SAVE_CSV:
    print()
    print("Saved CSV to:")
    print(CSV_PATH)

print()
print("Successfully processed indices:")
print(processed_indices[valid_rows])

Selected 401 matrices.
Matrix/spectrum size: 2600
Using 8 BLAS/LAPACK threads per SVD.
[   1/401] E_0: shape=(2600, 2600), nnz=246094, sigma_max=6.919141e+01, sigma_min=4.892228e-02, endpoint=-5.391e-12
[   2/401] E_1: shape=(2600, 2600), nnz=246094, sigma_max=6.923721e+01, sigma_min=4.893667e-02, endpoint=-5.569e-12
[   3/401] E_2: shape=(2600, 2600), nnz=246094, sigma_max=6.928307e+01, sigma_min=4.895300e-02, endpoint=-5.962e-12
